In [ ]:
import sys
sys.path.insert(0, '/home/simplexity/cyt/pinnsformer-main')

import time

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import random
from torch.optim import LBFGS
from tqdm import tqdm
import scipy.io
import json

from util import *
from model.pinn import PINNs
from model.pinnsformer import PINNsformer

# Table 1 Convection PINNsFormer Reproduction

This copied notebook keeps the paper/Table 1 full setting for the Convection PINNsFormer experiment:

- training grid: `51 x 51`
- test grid: `101 x 101`
- pseudo sequence length: `k = 5`
- sequence step: `Delta t = 1e-4`
- model: `d_model=32`, `d_hidden=512`, `heads=2`, `encoder/decoder layers=1`
- optimizer: `LBFGS(line_search_fn="strong_wolfe")`
- iterations: `1000`

Target from paper Table 1: `Loss=3.7e-5`, `rMAE=0.023`, `rRMSE=0.027`.


In [ ]:
seed = 0
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

device = 'cuda:0'

In [ ]:
# Train PINNsformer
TABLE1_CONFIG = {
    'problem': 'convection',
    'model': 'PINNsFormer',
    'grid_size': 51,
    'test_grid_size': 101,
    'sequence_length': 5,
    'sequence_step': 1e-4,
    'd_model': 32,
    'd_hidden': 512,
    'heads': 2,
    'encoder_layers': 1,
    'optimizer': 'LBFGS strong_wolfe',
    'epochs': 1000,
    'paper_table1_loss': 3.7e-5,
    'paper_table1_rMAE': 0.023,
    'paper_table1_rRMSE': 0.027,
}
print(TABLE1_CONFIG)
res, b_left, b_right, b_upper, b_lower = get_data([0,2*np.pi], [0,1], 51, 51)
res_test, _, _, _, _ = get_data([0,2*np.pi], [0,1], 101, 101)

res = make_time_sequence(res, num_step=5, step=1e-4)
b_left = make_time_sequence(b_left, num_step=5, step=1e-4)
b_right = make_time_sequence(b_right, num_step=5, step=1e-4)
b_upper = make_time_sequence(b_upper, num_step=5, step=1e-4)
b_lower = make_time_sequence(b_lower, num_step=5, step=1e-4)

res = torch.tensor(res, dtype=torch.float32, requires_grad=True).to(device)
b_left = torch.tensor(b_left, dtype=torch.float32, requires_grad=True).to(device)
b_right = torch.tensor(b_right, dtype=torch.float32, requires_grad=True).to(device)
b_upper = torch.tensor(b_upper, dtype=torch.float32, requires_grad=True).to(device)
b_lower = torch.tensor(b_lower, dtype=torch.float32, requires_grad=True).to(device)

x_res, t_res = res[:,:,0:1], res[:,:,1:2]
x_left, t_left = b_left[:,:,0:1], b_left[:,:,1:2]
x_right, t_right = b_right[:,:,0:1], b_right[:,:,1:2]
x_upper, t_upper = b_upper[:,:,0:1], b_upper[:,:,1:2]
x_lower, t_lower = b_lower[:,:,0:1], b_lower[:,:,1:2]

def init_weights(m):
    if isinstance(m, nn.Linear):
        torch.nn.init.xavier_uniform(m.weight)
        m.bias.data.fill_(0.01)

In [ ]:
model = PINNsformer(d_out=1, d_hidden=512, d_model=32, N=1, heads=2).to(device)

model.apply(init_weights)
optim = LBFGS(model.parameters(), line_search_fn='strong_wolfe')

print(model)
print(get_n_params(model))

In [ ]:
loss_track = []
start_time = time.time()

for i in tqdm(range(1000)):
    def closure():
        pred_res = model(x_res, t_res)
        pred_left = model(x_left, t_left)
        pred_right = model(x_right, t_right)
        pred_upper = model(x_upper, t_upper)
        pred_lower = model(x_lower, t_lower)

        u_x = torch.autograd.grad(pred_res, x_res, grad_outputs=torch.ones_like(pred_res), retain_graph=True, create_graph=True)[0]
        u_t = torch.autograd.grad(pred_res, t_res, grad_outputs=torch.ones_like(pred_res), retain_graph=True, create_graph=True)[0]

        loss_res = torch.mean((u_t + 50 * u_x) ** 2)
        loss_bc = torch.mean((pred_upper - pred_lower) ** 2)
        loss_ic = torch.mean((pred_left[:,0] - torch.sin(x_left[:,0])) ** 2)

        loss_track.append([loss_res.item(), loss_bc.item(), loss_ic.item()])

        loss = loss_res + loss_bc + loss_ic
        optim.zero_grad()
        loss.backward()
        return loss
    
    optim.step(closure)

elapsed = time.time() - start_time
h = int(elapsed // 3600)
m = int((elapsed % 3600) // 60)
s = int(elapsed % 60)
print(f'Training finished. Total time: {h}h {m}min {s}s')

In [ ]:
print('Loss Res: {:4f}, Loss_BC: {:4f}, Loss_IC: {:4f}'.format(loss_track[-1][0], loss_track[-1][1], loss_track[-1][2]))
print('Train Loss: {:4f}'.format(np.sum(loss_track[-1])))

torch.save(model.state_dict(), 'convection_pinnsformer_table1_repro.pt')

In [ ]:
# Visualize PINNsformer
res_test = make_time_sequence(res_test, num_step=5, step=1e-4) 
res_test = torch.tensor(res_test, dtype=torch.float32, requires_grad=True).to(device)
x_test, t_test = res_test[:,:,0:1], res_test[:,:,1:2]

with torch.no_grad():
    pred = model(x_test, t_test)[:,0:1]
    pred = pred.cpu().detach().numpy()

pred = pred.reshape(101,101)

mat = scipy.io.loadmat('./convection.mat')
u = mat['u'].reshape(101,101)

rl1 = np.sum(np.abs(u-pred)) / np.sum(np.abs(u))
rl2 = np.sqrt(np.sum((u-pred)**2) / np.sum(u**2))

print('relative L1 error: {:4f}'.format(rl1))
print('relative L2 error: {:4f}'.format(rl2))


metrics = {
    **TABLE1_CONFIG,
    'train_loss': float(np.sum(loss_track[-1])),
    'loss_res': float(loss_track[-1][0]),
    'loss_bc': float(loss_track[-1][1]),
    'loss_ic': float(loss_track[-1][2]),
    'rMAE_relative_L1': float(rl1),
    'rRMSE_relative_L2': float(rl2),
}
with open('convection_pinnsformer_table1_repro_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved metrics to convection_pinnsformer_table1_repro_metrics.json')
print(metrics)

plt.figure(figsize=(4,3))
plt.imshow(pred, extent=[0,np.pi*2,1,0], aspect='auto')
plt.xlabel('x')
plt.ylabel('t')
plt.title('Predicted u(x,t)')
plt.colorbar()
plt.tight_layout()
plt.savefig('./convection_pinnsformer_table1_repro_pred.png')
plt.show()

In [ ]:
plt.figure(figsize=(4,3))
plt.imshow(u, extent=[0,np.pi*2,1,0], aspect='auto')
plt.xlabel('x')
plt.ylabel('t')
plt.title('Exact u(x,t)')
plt.colorbar()
plt.tight_layout()
plt.savefig('./convection_table1_repro_exact.png')
plt.show()

In [ ]:
plt.figure(figsize=(4,3))
plt.imshow(np.abs(pred - u), extent=[0,np.pi*2,1,0], aspect='auto')
plt.xlabel('x')
plt.ylabel('t')
plt.title('Absolute Error')
plt.colorbar()
plt.tight_layout()
plt.savefig('./convection_pinnsformer_table1_repro_error.png')
plt.show()

## 复现判据

运行结束后查看 `convection_pinnsformer_table1_repro_metrics.json`。若接近论文 Table 1，应满足：

- `train_loss` 接近 `3.7e-5`
- `rMAE_relative_L1` 接近 `0.023`
- `rRMSE_relative_L2` 接近 `0.027`

如果 `loss_bc` 长期停在 `1e-2` 量级，通常表示周期边界条件没有收敛，属于 failure-mode 式坏解；建议更换 seed 或使用原始 notebook 环境重跑。
